In [2]:
import random
import math
import hashlib
import struct
from typing import Callable, Any

In [3]:
class IntegerHash:
    def __init__(self):
        self._seed = random.randint(0, 2**32 - 1)

    def __call__(self, item: Any) -> int:
        text = str(item) + str(self._seed)
        hash_bytes = hashlib.md5(text.encode('utf-8')).digest()
        return struct.unpack('>I', hash_bytes[:4])[0]

In [4]:
class HLLSketch:
    def __init__(self, b: int, hashf: Callable[[Any], int]):
        self._b = b
        self._m = 1 << b
        self._hash = hashf
        self._registers = [0] * self._m

        self._alpha_hl = 0.39701
        self._alpha_hll = HLLSketch.get_alpha_hll(self._m)

    def add(self, item: Any):
        h = self._hash(item)
        j = h & (self._m - 1)
        w = h >> self._b
        self._registers[j] = max(self._registers[j], self._rho(w))

    def estimate_hyperlog(self) -> float:
        """
        Arithmetic Mean
        """
        arithmetic_mean = sum(self._registers) / self._m
        return self._alpha_hl * self._m * (2 ** arithmetic_mean)

    def estimate_hyperloglog(self) -> float:
        """
        Harmonic Mean
        """
        sum_inverse_pow2 = sum(2 ** (-r) for r in self._registers)
        if sum_inverse_pow2 == 0:
            return 0.0

        return self._alpha_hll * (self._m ** 2) / sum_inverse_pow2

    def n_empty_registers(self) -> int:
        return self._registers.count(0)

    def _rho(self, w: int) -> int:
        if w == 0:
            return 32 - self._b + 1
        return (w & -w).bit_length()

    @staticmethod
    def get_alpha_hll(m: int) -> float:
        if m == 16:
            return 0.673
        elif m == 32:
            return 0.697
        elif m == 64:
            return 0.709
        else:
            return 0.7213 / (1 + 1.079 / m)

In [6]:
b = 10
k = 1 << b
hashf = IntegerHash()

boundary = k * math.log(k)
test_cardinalities = [
    int(boundary * 0.25),
    int(boundary * 0.5),
    int(boundary),
    int(boundary * 2.0),
    int(boundary * 4.0)
]

print(f"Number of Registers: {k}")
print(f"Theoretical Boundry: ~ {boundary:.2f}")
print("-" * 80)
print(f"{'True N':<10} | {'HyperLog':<12} {'Err(%)':<8} | {'HyperLogLog':<12} {'Err(%)':<8} | {'Empty Registers':<10}")
print("-" * 80)

for n in test_cardinalities:
    sketch = HLLSketch(b=b, hashf=hashf)

    stream = list(range(n))
    random.shuffle(stream)
    for item in stream:
        sketch.add(item)

    est_hl = sketch.estimate_hyperlog()
    est_hll = sketch.estimate_hyperloglog()

    err_ll = abs(est_hl - n) / n * 100
    err_hll = abs(est_hll - n) / n * 100
    zeros = sketch.n_empty_registers()

    print(f"{n:<10} | {est_hl:<12.2f} {err_ll:<8.2f} | {est_hll:<12.2f} {err_hll:<8.2f} | {zeros:<10}")

print("-" * 80)

Number of Registers: 1024
Theoretical Boundry: ~ 7097.83
--------------------------------------------------------------------------------
True N     | HyperLog     Err(%)   | HyperLogLog  Err(%)   | Empty Registers
--------------------------------------------------------------------------------
1774       | 1882.17      6.10     | 1992.90      12.34    | 165       
3548       | 3701.18      4.32     | 3614.22      1.87     | 33        
7097       | 7292.95      2.76     | 7147.91      0.72     | 0         
14195      | 15407.96     8.54     | 14811.69     4.34     | 0         
28391      | 29972.44     5.57     | 28530.66     0.49     | 0         
--------------------------------------------------------------------------------
